# TrafficGuard AI — Model Training Notebook

This notebook trains the **rider (person on two-wheeler)** detector used by TrafficGuard AI, and shows how to install
the **helmet** detector. It is designed to run on **Kaggle** (free GPU, dataset already mounted) but works on Colab or
locally with the paths changed.

**Reference:** [Motorbike and Helmet YOLO Detection](https://www.kaggle.com/code/stpeteishii/motorbike-and-helmet-yolo-detection) by stpeteishii.
That notebook runs two *pretrained* Darknet YOLOv3 models with OpenCV DNN, taken from two datasets by savanagrawal:

| Dataset | Contents | Classes |
|---|---|---|
| `detect-person-on-motorbike-or-scooter` | `dataset/obj/*.jpg` + YOLO `.txt` labels (one box per image, with `__flip` augmented copies), `coco.names`, YOLOv3 weights/cfg | 1 class: person on a two-wheeler |
| `helmet-detection-yolov3` | `yolov3-helmet.cfg`, `yolov3-helmet.weights`, `helmet.names` — **no images** | 1 class: `Helmet` |

So the reference data supports **rider detection** (trainable) and **helmet detection** (pretrained weights only).
Vehicles, persons and traffic lights come from the COCO-pretrained YOLO11 model.

**On Kaggle:** *Add Input* → add both datasets above → enable the **GPU** accelerator → *Run all*.

In [ ]:
# 1. Get the project code and dependencies
!git clone -q https://github.com/muhammedaabirturab/traffic-camera-detection.git
%cd traffic-camera-detection
!pip install -q -r requirements.txt

In [ ]:
# 2. Locate the datasets (adjust if running outside Kaggle)
from pathlib import Path
RIDER_SRC  = Path('/kaggle/input/detect-person-on-motorbike-or-scooter')
HELMET_SRC = Path('/kaggle/input/helmet-detection-yolov3')
print('rider dataset :', RIDER_SRC.exists(), '| helmet weights:', HELMET_SRC.exists())

## 3. Inspect the dataset before training
Check — rather than assume — the class names, number of images, label format and the flipped duplicates.

In [ ]:
import collections, random
obj = RIDER_SRC / 'dataset' / 'obj'
imgs = sorted(obj.glob('*.jpg'))
labels = sorted(obj.glob('*.txt'))
print('images:', len(imgs), '| label files:', len(labels))
print('flipped copies:', sum('__flip' in p.stem for p in imgs))
for nf in RIDER_SRC.rglob('*.names'):
    print(nf.name, '->', nf.read_text().split())
cls_counts, boxes_per_img = collections.Counter(), collections.Counter()
for lf in labels:
    rows = [r.split() for r in lf.read_text().splitlines() if r.strip()]
    boxes_per_img[len(rows)] += 1
    cls_counts.update(r[0] for r in rows)
print('class index counts:', dict(cls_counts))
print('boxes per image   :', dict(sorted(boxes_per_img.items())))

In [ ]:
import cv2, matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, p in zip(axes.flat, random.Random(1).sample(imgs, 8)):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    for r in p.with_suffix('.txt').read_text().splitlines():
        c, x, y, bw, bh = map(float, r.split()[:5])
        cv2.rectangle(im, (int((x-bw/2)*w), int((y-bh/2)*h)), (int((x+bw/2)*w), int((y+bh/2)*h)), (0, 255, 255), 3)
    ax.imshow(im); ax.set_title(p.name); ax.axis('off')
plt.tight_layout()

## 4. Prepare a leakage-free split
`prepare_dataset.py` validates every label line and keeps each image together with its `__flip` copy in the same split
(otherwise near-identical images would appear in both train and validation and inflate the metrics).
We name the class `rider` because each box marks a person seated on a two-wheeler.

In [ ]:
!python scripts/prepare_dataset.py --source {RIDER_SRC} --out datasets/rider --names rider --val 0.15 --test 0.10

## 5. Train (transfer learning from COCO-pretrained YOLO11n)
Increase `--epochs` / use `yolo11s.pt` for better accuracy if time allows. Early stopping (`--patience`) prevents over-training.

In [ ]:
!python scripts/train.py --data datasets/rider/data.yaml --role rider --model yolo11n.pt --epochs 60 --imgsz 640 --batch 32 --device 0

In [ ]:
# 6. Metrics measured on the held-out TEST split (written by train.py)
import json
m = json.load(open('models/metrics/rider.json'))
print(json.dumps(m['metrics'], indent=2))

In [ ]:
from IPython.display import Image, display
for p in m['plots']:
    if p.endswith(('results.png', 'BoxPR_curve.png', 'confusion_matrix.png')):
        display(Image(filename=f'models/metrics/{p}', width=900))

## 7. Helmet detector
**Option A (no training):** copy the reference YOLOv3 helmet model into `models/`. TrafficGuard loads Darknet
`.cfg`/`.weights` through OpenCV DNN automatically. (No images are provided with it, so its accuracy cannot be
measured here; the Model page will say so.)

**Option B:** train a YOLO11 helmet model on a helmet dataset *with images* (classes such as `With Helmet` / `Without Helmet`),
e.g. `prepare_dataset.py --format voc --class-map "With Helmet=helmet,Without Helmet=no_helmet"` then `train.py --role helmet`.

In [ ]:
import shutil
for f in ['yolov3-helmet.cfg', 'yolov3-helmet.weights', 'helmet.names']:
    if (HELMET_SRC / f).exists():
        shutil.copy(HELMET_SRC / f, 'models/' + f); print('copied', f)

## 8. Download the results
Download `models/rider.pt`, the `models/metrics/` folder and (Option A) the three helmet files, and place them in the
`models/` folder of your local clone. Then restart the server — the Model page will show the measured metrics.

In [ ]:
!cd models && zip -qr ../trafficguard_models.zip rider.pt metrics yolov3-helmet.cfg helmet.names && echo 'trafficguard_models.zip ready (helmet .weights is ~235 MB, download it separately)'